# Senv χ² profiles

This notebook produces three figures from `Senv_nue.csv` and `Senv_numu.csv`:

1. The electron-neutrino profile, `chi2_nue` versus `drhocpt`.
2. The muon-neutrino profile, `chi2_numu` versus `drhocpt`.
3. The total profile, `chi2_tot = chi2_nue + chi2_numu`, versus `drhocpt`.

The input directory is selected automatically from the most recently modified folder matching `IntSenv_*` under `/home/ydenizhernandez/P-NOTE/`. This accommodates the timestamp in the folder name.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Parent directory containing timestamped IntSenv_* folders
BASE_DIR = Path("/home/ydenizhernandez/P-NOTE")

# Select the newest available IntSenv_* directory
candidate_dirs = [p for p in BASE_DIR.glob("IntSenv_*") if p.is_dir()]
if not candidate_dirs:
    raise FileNotFoundError(f"No IntSenv_* folders found in {BASE_DIR}")

DATA_DIR = max(candidate_dirs, key=lambda p: p.stat().st_mtime)

nue_file = DATA_DIR / "Senv_nue.csv"
numu_file = DATA_DIR / "Senv_numu.csv"

print(f"Using input directory: {DATA_DIR}")
print(f"νe file exists:  {nue_file.exists()}")
print(f"νμ file exists:  {numu_file.exists()}")

if not nue_file.exists() or not numu_file.exists():
    raise FileNotFoundError("Expected Senv_nue.csv and Senv_numu.csv in the selected directory.")

In [ ]:
# Load the two profiles
df_nue = pd.read_csv(nue_file)
df_numu = pd.read_csv(numu_file)

# Clean column names in case the CSV headers contain surrounding whitespace
df_nue.columns = df_nue.columns.str.strip()
df_numu.columns = df_numu.columns.str.strip()

required_columns = {"drhocpt", "chi2_flv"}
for filename, df in [("Senv_nue.csv", df_nue), ("Senv_numu.csv", df_numu)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"{filename} is missing required column(s): {sorted(missing)}")

# Keep only the columns needed for the profiles, convert to numeric, and sort
df_nue = df_nue[["drhocpt", "chi2_flv"]].apply(pd.to_numeric, errors="coerce").dropna()
df_numu = df_numu[["drhocpt", "chi2_flv"]].apply(pd.to_numeric, errors="coerce").dropna()

df_nue = df_nue.sort_values("drhocpt").rename(columns={"chi2_flv": "chi2_nue"})
df_numu = df_numu.sort_values("drhocpt").rename(columns={"chi2_flv": "chi2_numu"})

display(df_nue.head())
display(df_numu.head())

## 1. Electron-neutrino profile

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_nue["drhocpt"], df_nue["chi2_nue"], marker="o", label=r"$\chi^2_{\nu_e}$")
ax.set_xlabel(r"$\delta\rho_c$ (%)")
ax.set_ylabel(r"$\chi^2$")
ax.set_title(r"$\chi^2_{\nu_e}$ profile — 100 Mton·year")
ax.legend(title=r"$\chi^2_{\nu_e}$")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Muon-neutrino profile

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_numu["drhocpt"], df_numu["chi2_numu"], marker="o", label=r"$\chi^2_{\nu_\mu}$")
ax.set_xlabel(r"$\delta\rho_c$ (%)")
ax.set_ylabel(r"$\chi^2$")
ax.set_title(r"$\chi^2_{\nu_\mu}$ profile — 100 Mton·year")
ax.legend(title=r"$\chi^2_{\nu_\mu}$")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3. Total profile: electron + muon neutrinos

The two profiles are combined by matching rows with the same `drhocpt` value. If the two CSV files use different `drhocpt` grids, only their common grid points are included in the total profile.

In [ ]:
# Combine the profiles at matching drhocpt values
df_tot = pd.merge(
    df_nue[["drhocpt", "chi2_nue"]],
    df_numu[["drhocpt", "chi2_numu"]],
    on="drhocpt",
    how="inner",
    validate="one_to_one"
).sort_values("drhocpt")

if df_tot.empty:
    raise ValueError(
        "The two CSV files have no matching drhocpt values. "
        "An interpolation step would be needed to calculate chi2_tot."
    )

df_tot["chi2_tot"] = df_tot["chi2_nue"] + df_tot["chi2_numu"]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_tot["drhocpt"], df_tot["chi2_tot"], marker="o", label=r"$\chi^2_{\mathrm{tot}}$")
ax.set_xlabel(r"$\delta\rho_c$ (%)")
ax.set_ylabel(r"$\chi^2$")
ax.set_title(r"$\chi^2_{\mathrm{tot}} = \chi^2_{\nu_e} + \chi^2_{\nu_\mu}$ profile — 100 Mton·year")
ax.legend(title=r"$\chi^2_{\mathrm{tot}}$")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

display(df_tot.head())